FAISS

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter


In [3]:
loader = TextLoader('speech_text.txt')
doc = loader.load()


In [5]:
text_splitter = CharacterTextSplitter(chunk_size=500,chunk_overlap=30)
docs = text_splitter.split_documents(doc)

Created a chunk of size 521, which is longer than the specified 500
Created a chunk of size 509, which is longer than the specified 500


In [7]:
print(docs[0])

page_content='Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as 'Bebo')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal granddaughter of actor Hari Shivdasani, and great-granddaughter of filmmaker Prithviraj Kapoor. The actors Rishi Kapoor and Neetu Kapoor are her uncle and aunt respectively, and actor Ranbir Kapoor is also her cousin.' metadata={'source': 'speech_text.txt'}


In [8]:
embeddings = OllamaEmbeddings(model="gemma2:2b")
db = FAISS.from_documents(docs,embeddings)

/var/folders/ck/tj_z0jk1249ggvdcqp5pk2200000gn/T/ipykernel_3566/4190531185.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="gemma2:2b")


In [14]:
query = "Who is Kareena Kapoor's mother?"
res = db.similarity_search(query)


In [15]:
print(res[0].page_content)

Describing herself as a naughty, spoilt child, Kapoor's exposure to films from a young age kindled her interest in acting;[5] she was particularly inspired by the work of actresses Nargis and Meena Kumari.[10] Despite her family background, her father disapproved of women entering films because he believed it conflicted with the traditional maternal duties and responsibility of women in the family.[11] This led to a conflict between her parents and they lived separately before reconciling in October 2007.[12] She was raised by her mother, who worked several jobs to support her daughters until Karisma debuted as an actress in 1991.[13] Although her father was not present for most her childhood, Kapoor remarked that he played an important role in her life.[13]


Query as a Retriever.

We can also convert the vector store into a Rectriver class. This allows us to easily use it in other Langchain methods, which largely works with retrievers.

In [16]:
retriever = db.as_retriever()
doc = retriever.invoke(query)
doc[0].page_content

"Describing herself as a naughty, spoilt child, Kapoor's exposure to films from a young age kindled her interest in acting;[5] she was particularly inspired by the work of actresses Nargis and Meena Kumari.[10] Despite her family background, her father disapproved of women entering films because he believed it conflicted with the traditional maternal duties and responsibility of women in the family.[11] This led to a conflict between her parents and they lived separately before reconciling in October 2007.[12] She was raised by her mother, who worked several jobs to support her daughters until Karisma debuted as an actress in 1991.[13] Although her father was not present for most her childhood, Kapoor remarked that he played an important role in her life.[13]"

Similarity search with score.

There are fAISS specific methods. One of them is similarity_search_with_score which returns not only documents but also distance scoe of query to them. Returned distance score is L2 distance (manhattan).Lower score is better

In [17]:
docs_and_score = db.similarity_search_with_score(query)
print(docs_and_score)

[(Document(id='dcf3f6bc-a4d9-47f7-8845-e5943bca55bf', metadata={'source': 'speech_text.txt'}, page_content="Describing herself as a naughty, spoilt child, Kapoor's exposure to films from a young age kindled her interest in acting;[5] she was particularly inspired by the work of actresses Nargis and Meena Kumari.[10] Despite her family background, her father disapproved of women entering films because he believed it conflicted with the traditional maternal duties and responsibility of women in the family.[11] This led to a conflict between her parents and they lived separately before reconciling in October 2007.[12] She was raised by her mother, who worked several jobs to support her daughters until Karisma debuted as an actress in 1991.[13] Although her father was not present for most her childhood, Kapoor remarked that he played an important role in her life.[13]"), 4286.986), (Document(id='d92f7d24-60d0-4c0c-b821-82cb8e87227e', metadata={'source': 'speech_text.txt'}, page_content="Bo

Saving and loading Vector Store db in local 

In [20]:
db.save_local("faiss_index") #it will get saved with 'faiss_index' name


In [21]:
new_db = FAISS.load_local("faiss_index",embeddings)

ValueError: The de-serialization relies loading a pickle file. Pickle files can be modified to deliver a malicious payload that results in execution of arbitrary code on your machine.You will need to set `allow_dangerous_deserialization` to `True` to enable deserialization. If you do this, make sure that you trust the source of the data. For example, if you are loading a file that you created, and know that no one else has modified the file, then this is safe to do. Do not set this to `True` if you are loading a file from an untrusted source (e.g., some random site on the internet.).

this abive error occurs in new version of faiss, to overcome this just add one parameter like given below

In [22]:
new_db = FAISS.load_local("faiss_index",embeddings,allow_dangerous_deserialization=True)

In [23]:
new_db.similarity_search(query)

[Document(id='dcf3f6bc-a4d9-47f7-8845-e5943bca55bf', metadata={'source': 'speech_text.txt'}, page_content="Describing herself as a naughty, spoilt child, Kapoor's exposure to films from a young age kindled her interest in acting;[5] she was particularly inspired by the work of actresses Nargis and Meena Kumari.[10] Despite her family background, her father disapproved of women entering films because he believed it conflicted with the traditional maternal duties and responsibility of women in the family.[11] This led to a conflict between her parents and they lived separately before reconciling in October 2007.[12] She was raised by her mother, who worked several jobs to support her daughters until Karisma debuted as an actress in 1991.[13] Although her father was not present for most her childhood, Kapoor remarked that he played an important role in her life.[13]"),
 Document(id='d92f7d24-60d0-4c0c-b821-82cb8e87227e', metadata={'source': 'speech_text.txt'}, page_content="Born on 21 Sep

CHROMA DB

In [24]:
from langchain_chroma import Chroma


In [25]:
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [26]:
loader = TextLoader('speech_text.txt')
doc = loader.load()

In [27]:
doc

[Document(metadata={'source': 'speech_text.txt'}, page_content='Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as \'Bebo\')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal granddaughter of actor Hari Shivdasani, and great-granddaughter of filmmaker Prithviraj Kapoor. The actors Rishi Kapoor and Neetu Kapoor are her uncle and aunt respectively, and actor Ranbir Kapoor is also her cousin.\n\nKareena and Karisma Kapoor with their mother Babita\nPictured with mother Babita (left) and sister Karisma. In an interview with journalist Vir Sanghvi, Kapoor stated that growing up with the two of them helped her become strong and independent.[5]\nAccording to Kapoor, her name "Kareena" was derived from the book Anna Karenina, which her mother read while she was pregnant with her.[6] Kapoor is of Punjabi

In [28]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=30)
split = text_splitter.split_documents(doc)

In [31]:
print(split[0].page_content)

Born on 21 September 1980 in Bombay (now Mumbai),[2] Kapoor (often informally referred to as 'Bebo')[3] is the younger daughter of Randhir Kapoor and Babita (née Shivdasani);[4] her elder sister Karisma is also an actress. Kapoor is the paternal granddaughter of actor and filmmaker Raj Kapoor, maternal granddaughter of actor Hari Shivdasani, and great-granddaughter of filmmaker Prithviraj Kapoor. The actors Rishi Kapoor and Neetu Kapoor are her uncle and aunt respectively, and actor Ranbir


In [43]:
embeddings = OllamaEmbeddings(model = "gemma2:2b")
#saving to disk with persist_directory command
db = Chroma.from_documents(split,embeddings,persist_directory="./chroma_db")

In [41]:
query = "Kareena was derived from the book"

In [44]:

docs_res= db.similarity_search(query)
print(docs_res[0].page_content)

According to Kapoor, her name "Kareena" was derived from the book Anna Karenina, which her mother read while she was pregnant with her.[6] Kapoor is of Punjabi descent on her father's side,[7] and on her mother's side she is of Sindhi and British descent.[8][9]


In [45]:
#laoding 
new_db2 = Chroma(persist_directory="./chroma_db",embedding_function=embeddings)

we can ceate retriever just like above we crearwd for faiss